In [1]:
import re

In [2]:
import json

In [3]:
import pickle

In [4]:
import tqdm

In [5]:
from dotenv import load_dotenv

In [6]:
load_dotenv()

False

In [7]:
import pandas as pd

In [8]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [9]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [10]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [11]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [12]:
with open("preds_wo_features.pkl", "rb") as f:
    preds_wo_features = pickle.load(f)

In [13]:
with open("preds_w_features.pkl", "rb") as f:
    preds_w_features = pickle.load(f)

In [14]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags_dict = pickle.load(f)

In [15]:
with open("artificial_profiles.json", "rb") as f:
    artificial_profiles = json.load(f)

In [16]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can evaluate, wherether or not a proposed project fits student interests, and explain.
Return strictly JSON. Take into account bio and interests of a student.
Example Input:
```
You are machine_learning_researcher student.
Your bio is 'A student, which enjoys solving machine learning problems'.
Your interests are ['machine_learning', deep_learning'].
You are proposed with a project named 'Рекомендательная система студенческих проектов' with corresponding tags ['machine_learning', 'deep_learning', 'recommender_systems'].
```
Exapmle Output:
```json
{'is_valid_recommendation': 1, 'explanation': 'The recommendaditon is valid in terms of semantric relevance and tags intersection'}
```
"""

In [17]:
def process(df):
    messages = []

    for row in df.iterrows():
        user_name = row[1].user_name
        user_bio = artificial_profiles[user_name]["bio"]
        user_tags = artificial_profiles[user_name]["tags"]

        item_title = row[1].item_title
        item_tags = titles_with_tags_dict[item_title]

        USER_MESSAGE = f"""
        You are {user_name} student.
        Your bio is '{user_bio}'.
        Your interests are {user_tags}.
        You are proposed with a project named '{item_title}' with corresponding tags {item_tags}.
        """

        messages.append({
            "user_name": user_name,
            "item_title": item_title,
            "shared_tags": len(set(user_tags).intersection(item_tags)) / len(item_tags),
            "message":
            (
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_MESSAGE},
            ),
            })

    data = []

    for row in tqdm.tqdm(messages):
        text = processor.apply_chat_template(
            row["message"], tokenize=False, add_generation_prompt=True, enable_thinking=False
        )

        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        outputs = model.generate(**inputs, max_new_tokens=1024)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        try:
            json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_str = response
            answer = json.loads(json_str)
        except Exception:
            answer = {"is_valid_recommendation": None}

        try:
            is_valid = answer.get("is_valid_recommendation", None)
            is_valid = int(is_valid)
            explanation = answer.get("explanation", None)
            data.append((row["user_name"], row["item_title"], row["shared_tags"], is_valid, explanation))
        except (TypeError, ValueError):
            pass

    return pd.DataFrame(data=data, columns=["user_name", "item_title", "shared_tags", "is_valid", "explanation"])

In [18]:
preds_wo_features_scores = process(preds_wo_features)

100%|██████████| 310/310 [27:56<00:00,  5.41s/it]


In [19]:
preds_w_features_scores = process(preds_w_features)

100%|██████████| 310/310 [28:16<00:00,  5.47s/it]


In [20]:
preds_wo_features_scores.is_valid.mean()

np.float64(0.8258064516129032)

In [21]:
preds_w_features_scores.is_valid.mean()

np.float64(0.8935483870967742)

In [22]:
pwofg = preds_wo_features_scores.groupby(by=["user_name"]).agg({"is_valid": "mean"}).reset_index(drop=False)
pwofg.columns = ["user_name", "mean_score"]
pwofg = pwofg.sort_values(by=["mean_score"], ascending=False).reset_index(drop=True)

In [23]:
pwofg

,user_name,mean_score
0,digital_marketing_and_media_strategy,1.0
1,data_driven_policy_analyst,1.0
2,cultural_humanities_researcher_media_studies,1.0
3,geopolitics_and_international_relations_expert,1.0
4,global_economics_and_geopolitics_analyst,1.0
5,historical_culture_researcher_media_specialist,1.0
6,international_relations_and_geopolitics_expert,1.0
7,political_science_international_relations_analyst,1.0
8,regional_studies_and_geopolitics_expert,1.0
9,strategic_management_consultant_and_leader,1.0


In [24]:
pwfg = preds_w_features_scores.groupby(by=["user_name"]).agg({"is_valid": "mean"}).reset_index(drop=False)
pwfg.columns = ["user_name", "mean_score"]
pwfg = pwfg.sort_values(by=["mean_score"], ascending=False).reset_index(drop=True)

In [25]:
pwfg

,user_name,mean_score
0,cultural_and_media_anthropologist,1.0
1,data_driven_policy_analyst,1.0
2,cultural_humanities_researcher_media_studies,1.0
3,digital_marketing_and_media_strategy,1.0
4,global_economics_and_geopolitics_analyst,1.0
5,historical_culture_researcher_media_specialist,1.0
6,international_relations_and_geopolitics_expert,1.0
7,geopolitics_and_international_relations_expert,1.0
8,political_science_international_relations_analyst,1.0
9,regional_studies_and_geopolitics_expert,1.0


In [26]:
for explanation in preds_wo_features_scores.head(8).explanation.to_list():
  print(explanation)

The recommendation is valid in terms of semantic relevance and tags intersection. The project 'Цифровой универсальный комментарий' aligns with the student's interests in NLP, computer linguistics, and software engineering, all of which are directly covered in their interests and bio.
The project 'Учебный ассистент по курсу "Бухгалтерский учет"' aligns well with the student's interests in AI, NLP, and educational technology. The tags include 'ai' and 'educational_technology', which are directly relevant to the student's focus on intelligent systems and adaptive learning platforms. Additionally, the project involves a student assistant, which connects to their interest in chatbots and AI design. Although 'accounting' is a specific domain, the integration of AI and NLP in an educational context fits within the student's broader expertise and interests.
The project is valid in terms of semantic relevance and tags intersection. The student's interests in NLP, machine learning, programming l

In [27]:
for explanation in preds_wo_features_scores.tail(8).explanation.to_list():
  print(explanation)

The project is valid in terms of semantic relevance and tags intersection. The student's interests in market analysis, economics, and data analysis align well with the project's focus on market research in the food industry. Additionally, the student's expertise in strategic management and business strategy supports a thorough analysis of market dynamics, making this a relevant and suitable project.
The project is valid in terms of semantic relevance and tags intersection. The student's interests in strategic management, corporate strategy, data analysis, and AI align with the economic and financial analysis aspects of the project, particularly in the context of monetary policy and central banking. Although the focus is on macroeconomics, the student's background in strategic and data-driven decision-making supports a strong analytical approach to the topic.
The project is valid in terms of semantic relevance and tags intersection. The proposed project on EAEU integration from a trade 

In [28]:
for explanation in preds_w_features_scores.head(8).explanation.to_list():
  print(explanation)

The project 'Цифровой универсальный комментарий' aligns well with the student's interests in NLP, natural language processing, and software engineering. The tags 'natural_language_processing', 'nlp', and 'computer_linguistics' directly match the student's interests, and the involvement of software engineering fits their background in platform development and programming. The project is semantically relevant and intersects with the student's expertise and passions.
The project is valid in terms of semantic relevance and tags intersection. The student's interests in NLP, machine learning, programming language, and educational technology align well with the project's focus on a multi-agent intelligent learning system for JavaScript. The tags 'machine_learning', 'programming_language', and 'educational_technology' are directly relevant to the student's bio and interests, especially given their interest in adaptive learning platforms and dialogue systems.
The project 'Учебный ассистент по к

In [29]:
for explanation in preds_w_features_scores.tail(8).explanation.to_list():
  print(explanation)

The project is valid in terms of semantic relevance and tags intersection. The proposed project focuses on international business strategy and brand management in a global market (China), which aligns well with the student's interests in strategic management, business strategy, international business, and cross-cultural marketing. The tags 'business_strategy' and 'international_marketing' are directly relevant to the student's expertise and interests.
The project is valid in terms of semantic relevance and tags intersection. The proposed project involves business strategy and communication strategy, which align with the student's interests in business_strategy, strategic_management, and management. Although the focus is on regional marketing, it falls within the broader scope of business strategy and organizational communication, supported by the student's expertise in corporate strategy and data analysis.
The project is valid in terms of semantic relevance and tags intersection. The s

In [30]:
with open("preds_wo_features_scores.pkl", "wb") as f:
  pickle.dump(preds_wo_features_scores, f)

In [31]:
with open("preds_w_features_scores.pkl", "wb") as f:
  pickle.dump(preds_w_features_scores, f)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, summarize information related to project recommendations fit between to algorithms. Say, which one is better and explain in details.
Your input:
```[{'algo_name': 'lightfm_without_features', 'data': {'overall_accuracy': 0.5, 'eplanations': ['the recommendation is good', 'the recommendations is bad']}}, {'algo_name': 'lightfm_with_features', 'data': {'overall_accuracy': 1.0, 'eplanations': ['the recommendation is excellent', 'the recommendations is good']}}]```
Your output:
```'lightfm_with_features' better in terms of overall accuracy, semantic relevance and tags fit```
"""

In [38]:
USER_MESSAGE = json.dumps([
    {"algo_name": "lightfm_without_features", "overall_accuracy": preds_wo_features_scores.is_valid.mean(), "explanations": preds_wo_features_scores.explanation.to_list()},
    {"algo_name": "lightfm_with_features", "overall_accuracy": preds_w_features_scores.is_valid.mean(), "explanations": preds_w_features_scores.explanation.to_list()},
], ensure_ascii=False)

message = (
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE},
    )

text = processor.apply_chat_template(
    message, tokenize=False, add_generation_prompt=True, enable_thinking=False
)

inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

In [39]:
print(response)

`lightfm_with_features` is better in terms of overall accuracy, semantic relevance, and tags fit.  

### Detailed Explanation:

1. **Overall Accuracy**:
   - `lightfm_with_features` achieves an **overall accuracy of 0.8935**, which is **higher** than `lightfm_without_features` (0.8258).  
   - This indicates that the algorithm with feature integration performs better in predicting relevant project recommendations, resulting in fewer mismatches and more accurate alignment with the student's profile.  
   - The improved accuracy reflects a more robust understanding of the student’s interests, as features (such as tags, domain-specific keywords, and thematic overlap) enhance the model’s ability to capture nuanced preferences.

2. **Semantic Relevance**:
   - The explanations for `lightfm_with_features` consistently emphasize **strong semantic alignment** between the recommended projects and the student’s stated interests.  
   - Projects are described as aligning with core areas such as:
